# MATH4010 Final Project - Question 1


This notebook contains all three parts of Question 1:
- **Part 1(a)**: 2D area estimation (A₂ and B₂)
- **Part 1(b)**: 3D volume estimation (A₃ and B₃)
- **Part 1(c)**: 100D volume estimation (A₁₀₀ and B₁₀₀)


# Part 1(a) — 2D area Estimation

In 2-dimensional space ($n=2$), the "balls" are circles.

* $A_2$: A circle centered at $(0.5, 0.5)$ with a radius of $0.5$. It is inscribed within the unit square $[0, 1] \times [0, 1]$.
* $B_2$: A circle centered at the origin $(0, 0)$ with a radius of $1$. It is inscribed within the unit square $[-1, 1] \times [-1, 1]$.

In [14]:
import matplotlib
matplotlib.use('Agg')
# Configure consistent plot styling for all parts
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['legend.fontsize'] = 9
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['lines.linewidth'] = 2
plt.rcParams['lines.markersize'] = 6


In [15]:
import numpy as np
import matplotlib.patches as patches
import math

# Single RNG for the entire notebook (all Monte Carlo uses this)
RNG_SEED = 42
np.random.seed(RNG_SEED)
RNG = np.random.default_rng(RNG_SEED)


# Visualize $A_2$ and $B_2$

In [16]:
import os
os.makedirs('1', exist_ok=True)

# --- A2 ---
fig, ax = plt.subplots(figsize=(6, 6))
circle_a2 = patches.Circle((0.5, 0.5), 0.5, edgecolor='#1f77b4', facecolor='#1f77b4', alpha=0.5, label='$A_2$')
unit_box_a2 = patches.Rectangle((0, 0), 1, 1, fill=False, edgecolor='green', linestyle='--', linewidth=3, label='Box $[0, 1]^2$')
ax.add_patch(circle_a2)
ax.add_patch(unit_box_a2)
ax.plot(0.5, 0.5, 'ko', markersize=6)
ax.annotate('$(0.5, 0.5)$', (0.5, 0.5), textcoords="offset points", xytext=(-5, -25), ha='center', fontsize=22)
ax.set_xlim(-0.5, 1.5)
ax.set_ylim(-0.5, 1.5)
ax.set_aspect('equal')
ax.grid(True, linestyle=':', alpha=0.7)
ax.axhline(0, color='black', linewidth=1)
ax.axvline(0, color='black', linewidth=1)
ax.set_title('Visualization of $A_2$', fontsize=18)
ax.tick_params(axis='both', labelsize=22)
ax.legend(fontsize=16, loc='upper right')
plt.tight_layout()
plt.savefig('1/vis_A2.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved 1/vis_A2.png")

# --- B2 ---
fig, ax = plt.subplots(figsize=(6, 6))
circle_b2 = patches.Circle((0, 0), 1, edgecolor='#2ca02c', facecolor='#2ca02c', alpha=0.3, label='$B_2$')
unit_box_b2 = patches.Rectangle((-1, -1), 2, 2, fill=False, edgecolor='purple', linestyle='--', linewidth=2, label='Box $[-1, 1]^2$')
ax.add_patch(circle_b2)
ax.add_patch(unit_box_b2)
ax.plot(0, 0, 'ko', markersize=6)
ax.annotate('$(0, 0)$', (0, 0), textcoords="offset points", xytext=(10, -30), fontsize=22)
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')
ax.grid(True, linestyle=':', alpha=0.7)
ax.axhline(0, color='black', linewidth=1)
ax.axvline(0, color='black', linewidth=1)
ax.set_title('Visualization of $B_2$', fontsize=18)
ax.tick_params(axis='both', labelsize=22)
ax.legend(fontsize=16, loc='upper right')
plt.tight_layout()
plt.savefig('1/vis_B2.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved 1/vis_B2.png")


Saved 1/vis_A2.png
Saved 1/vis_B2.png


# Area of $A_2$ and $B_2$

The area of a 2-dimensional ball (a circle) is calculated using the formula:

$$Area = \pi r^2$$

The area of the 2 circles:

$$Area(A_2) = \pi \left(\frac{1}{2}\right)^2 = \frac{\pi}{4}$$

$$Area(B_2) = \pi (1)^2 = \pi$$

In [17]:
area_a2 = math.pi * (0.5)**2
area_b2 = math.pi * (1.0)**2
print(f"Exact Area of A2: {area_a2} (π/4)")
print(f"Exact Area of B2: {area_b2} (π)")


Exact Area of A2: 0.7853981633974483 (π/4)
Exact Area of B2: 3.141592653589793 (π)


# Metrics for Evaluation

Before implementing the algorithm, we define the metrics used to evaluate the Monte Carlo estimates.

## Primary Metrics

### 1. Estimate ($\hat{A}$)
The Monte Carlo estimate of the area:
$$\hat{A} = |\Omega| \cdot \frac{M}{N}$$
where $|\Omega|$ is the bounding box volume, $M$ is the number of hits, and $N$ is the total samples.

### 2. Hit Rate ($\hat{p}$)
The proportion of samples that fall inside the shape:
$$\hat{p} = \frac{M}{N}$$
This is an unbiased estimator of the true probability $p = A_{\text{true}} / |\Omega|$. The hit rate serves as a diagnostic metric - it should match the theoretical probability.

### 3. Absolute Error
The absolute difference between estimate and true value:
$$\text{Abs Error} = |\hat{A} - A_{\text{true}}|$$
This shows the magnitude of error in the same units as the area.

### 4. Relative Error (%)
The error as a percentage of the true value:
$$\text{Rel Error} = \frac{|\hat{A} - A_{\text{true}}|}{A_{\text{true}}} \times 100\%$$
This metric is dimension-independent and allows comparison across different shapes and dimensions.

## Statistical Metrics

### 5. Standard Error (SE)
The theoretical standard deviation of the estimator:
$$\text{SE} = |\Omega| \cdot \sqrt{\frac{\hat{p}(1-\hat{p})}{N}}$$
This quantifies the uncertainty in our estimate without requiring multiple trials. Note that SE decreases as $O(1/\sqrt{N})$, meaning we need 4× more samples to halve the error.

### 6. 95% Confidence Interval
Using the Wald interval (valid for large $N$ and $p$ not too close to 0 or 1):
$$\text{CI}_{95\%} = \hat{A} \pm 1.96 \cdot \text{SE}$$
This interval contains the true area with approximately 95% probability, providing a range of plausible values.

# Implementation

In [18]:
# Monte Carlo Hit-or-Miss Implementation
from typing import Tuple
import pandas as pd

TRUE_AREA_A2 = np.pi / 4
TRUE_AREA_B2 = np.pi

def monte_carlo_area_2d(shape: str, N: int, rng: np.random.Generator) -> Tuple[float, int, float]:
    if shape == 'A2':
        points = rng.uniform(0, 1, (N, 2))
        distances_sq = np.sum((points - 0.5)**2, axis=1)
        inside = distances_sq <= 0.25
        box_volume = 1.0
    elif shape == 'B2':
        points = rng.uniform(-1, 1, (N, 2))
        distances_sq = np.sum(points**2, axis=1)
        inside = distances_sq <= 1.0
        box_volume = 4.0
    else:
        raise ValueError(f"Unknown shape: {shape}. Use 'A2' or 'B2'.")
    M = np.sum(inside)
    area_estimate = box_volume * (M / N)
    return area_estimate, int(M), box_volume

def calculate_metrics(estimate, true_value, M, N, box_volume):
    p_hat = M / N
    abs_error = abs(estimate - true_value)
    rel_error = (abs_error / true_value) * 100
    if M == 0 or M == N:
        std_error = 0.0
    else:
        std_error = box_volume * np.sqrt(p_hat * (1 - p_hat) / N)
    z_score = 1.96
    ci_lower = estimate - z_score * std_error
    ci_upper = estimate + z_score * std_error
    return {
        'N': N, 'M': M, 'Hit_Rate': p_hat, 'Estimate': estimate,
        'True_Value': true_value, 'Abs_Error': abs_error,
        'Rel_Error_%': rel_error, 'Std_Error': std_error,
        'CI_Lower': ci_lower, 'CI_Upper': ci_upper
    }

N = 10_000_000
print(f"MONTE CARLO AREA ESTIMATION - PART 1(a)  (N = {N:,})\n")

est_A2, M_A2, box_A2 = monte_carlo_area_2d('A2', N, RNG)
met_A2 = calculate_metrics(est_A2, TRUE_AREA_A2, M_A2, N, box_A2)

est_B2, M_B2, box_B2 = monte_carlo_area_2d('B2', N, RNG)
met_B2 = calculate_metrics(est_B2, TRUE_AREA_B2, M_B2, N, box_B2)

for label, m in [('A_2', met_A2), ('B_2', met_B2)]:
    print(f"{label}: Estimate={m['Estimate']:.6f}, M={m['M']:,}, "
          f"RelErr={m['Rel_Error_%']:.4f}%, SE={m['Std_Error']:.6f}, "
          f"95%CI=[{m['CI_Lower']:.6f}, {m['CI_Upper']:.6f}]")


MONTE CARLO AREA ESTIMATION - PART 1(a)  (N = 10,000,000)

A_2: Estimate=0.785432, M=7,854,324, RelErr=0.0044%, SE=0.000130, 95%CI=[0.785178, 0.785687]
B_2: Estimate=3.141413, M=7,853,532, RelErr=0.0057%, SE=0.000519, 95%CI=[3.140395, 3.142431]


# Part 1(b): 3D Volume estimation


Visualization of A3 and B3:
- A3: Sphere centered at (1/2, 1/2, 1/2), radius = 1/2. This is perfectly inscribed within the unit cube ([0, 1])^3
- B3: Unit sphere centered at the origin (0, 0, 0) with radius = 1, contained within a larger bounding box

In [19]:
import os
os.makedirs('1', exist_ok=True)
import plotly.graph_objects as go
def get_sphere_data(center, radius, name, color):

    u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
    x = radius * np.cos(u) * np.sin(v) + center[0]
    y = radius * np.sin(u) * np.sin(v) + center[1]
    z = radius * np.cos(v) + center[2]
    return go.Surface(x=x, y=y, z=z, name=name, colorscale=[[0, color], [1, color]], showscale=False, opacity=0.8)
fig = go.Figure(data=[
    get_sphere_data([0, 0, 0], 1, "B3", "salmon"),
    get_sphere_data([0.5, 0.5, 0.5], 0.5, "A3", "cyan")
])
fig.update_layout(title="3D Visualization of A3 and B3", scene=dict(aspectmode='data'))
fig.write_html('1/vis_A3_B3.html')
print("Saved 1/vis_A3_B3.html (interactive)")


Saved 1/vis_A3_B3.html (interactive)


Calculate Volume using Normal calculus equation:

$$V_n(R) = \frac{\pi^{n/2}}{\Gamma(\frac{n}{2} + 1)} R^n$$

For $n=3$, this simplifies to the familiar formula $V_3 = \frac{4}{3}\pi R^3$.

In [20]:
# Exact volumes using the formula V = (4/3)πr³
# A3: Sphere with radius 0.5 centered at (0.5, 0.5, 0.5)
r_A3 = 0.5
true_A3 = (4/3) * np.pi * r_A3**3
print(f"Exact Volume A3: {true_A3:.6f} (π/6)")
# B3: Sphere with radius 1.0 centered at origin
r_B3 = 1.0
true_B3 = (4/3) * np.pi * r_B3**3
print(f"Exact Volume B3: {true_B3:.6f} (4π/3)")
# Store for use in implementation
TRUE_VOLUME_A3 = true_A3
TRUE_VOLUME_B3 = true_B3


Exact Volume A3: 0.523599 (π/6)
Exact Volume B3: 4.188790 (4π/3)


In [21]:
# Monte Carlo Volume Estimation - 3D
def monte_carlo_volume_3d(shape: str, N: int, rng: np.random.Generator) -> Tuple[float, int, float]:
    if shape == 'A3':
        points = rng.uniform(0, 1, (N, 3))
        distances_sq = np.sum((points - 0.5)**2, axis=1)
        inside = distances_sq <= 0.25
        box_volume = 1.0
    elif shape == 'B3':
        points = rng.uniform(-1, 1, (N, 3))
        distances_sq = np.sum(points**2, axis=1)
        inside = distances_sq <= 1.0
        box_volume = 8.0
    else:
        raise ValueError(f"Unknown shape: {shape}. Use 'A3' or 'B3'.")
    M = np.sum(inside)
    volume_estimate = box_volume * (M / N)
    return volume_estimate, int(M), box_volume

def calculate_metrics_3d(estimate, true_value, M, N, box_volume):
    p_hat = M / N
    abs_error = abs(estimate - true_value)
    rel_error = (abs_error / true_value) * 100
    if M == 0 or M == N:
        std_error = 0.0
    else:
        std_error = box_volume * np.sqrt(p_hat * (1 - p_hat) / N)
    z_score = 1.96
    ci_lower = estimate - z_score * std_error
    ci_upper = estimate + z_score * std_error
    return {
        'N': N, 'M': M, 'Hit_Rate': p_hat, 'Estimate': estimate,
        'True_Value': true_value, 'Abs_Error': abs_error,
        'Rel_Error_%': rel_error, 'Std_Error': std_error,
        'CI_Lower': ci_lower, 'CI_Upper': ci_upper
    }

N = 10_000_000
print(f"MONTE CARLO VOLUME ESTIMATION - PART 1(b)  (N = {N:,})\n")

est_A3, M_A3, box_A3 = monte_carlo_volume_3d('A3', N, RNG)
met_A3 = calculate_metrics_3d(est_A3, TRUE_VOLUME_A3, M_A3, N, box_A3)

est_B3, M_B3, box_B3 = monte_carlo_volume_3d('B3', N, RNG)
met_B3 = calculate_metrics_3d(est_B3, TRUE_VOLUME_B3, M_B3, N, box_B3)

for label, m in [('A_3', met_A3), ('B_3', met_B3)]:
    print(f"{label}: Estimate={m['Estimate']:.6f}, M={m['M']:,}, "
          f"RelErr={m['Rel_Error_%']:.4f}%, SE={m['Std_Error']:.6f}, "
          f"95%CI=[{m['CI_Lower']:.6f}, {m['CI_Upper']:.6f}]")


MONTE CARLO VOLUME ESTIMATION - PART 1(b)  (N = 10,000,000)

A_3: Estimate=0.523333, M=5,233,327, RelErr=0.0508%, SE=0.000158, 95%CI=[0.523023, 0.523642]
B_3: Estimate=4.187672, M=5,234,590, RelErr=0.0267%, SE=0.001264, 95%CI=[4.185196, 4.190148]


# Part 1(c): N-dimension estimation


We extend the hit-or-miss Monte Carlo method from Part 1(a) to 100 dimensions. The shapes are defined the same way as before:

- $A_n$: ball of radius $\tfrac{1}{2}$ centered at $\bigl(\tfrac{1}{2}, \ldots, \tfrac{1}{2}\bigr)$, inscribed in $[0,1]^n$.
- $B_n$: unit ball centered at the origin, enclosed by $[-1,1]^n$.

The exact volume of an $n$-dimensional ball of radius $R$ is
$$V_n(R) = \frac{\pi^{n/2}}{\,\Gamma\!\left(\frac{n}{2}+1\right)}\,R^n.$$

Lets implement the ground truth calculations for A and B

In [22]:
from scipy.special import loggamma

N_DIM = 100  # dimension for Part 1(c)


def volume_n_ball(n_dim, R=1.0):
    """Volume of an n-dimensional ball of radius R (stable log-space)."""
    log_V = (n_dim / 2) * np.log(np.pi) - loggamma(n_dim / 2 + 1.0) + n_dim * np.log(R)
    return float(np.exp(log_V))


# Exact volumes V(A_100), V(B_100)
TRUE_VOLUME_A100 = volume_n_ball(N_DIM, R=0.5)
TRUE_VOLUME_B100 = volume_n_ball(N_DIM, R=1.0)

# Bounding-box volumes: [0,1]^100 and [-1,1]^100
BOX_VOLUME_A100 = 1.0
BOX_VOLUME_B100 = 2.0 ** N_DIM

# Hit probability under uniform sampling in the bounding box
P_HIT_A100 = TRUE_VOLUME_A100 / BOX_VOLUME_A100
P_HIT_B100 = TRUE_VOLUME_B100 / BOX_VOLUME_B100

print(f"V(A_{N_DIM}) = {TRUE_VOLUME_A100:.6e}")
print(f"V(B_{N_DIM}) = {TRUE_VOLUME_B100:.6e}")
print(f"P(hit) in box:  A_{N_DIM} = {P_HIT_A100:.6e},  B_{N_DIM} = {P_HIT_B100:.6e}")


V(A_100) = 1.868182e-70
V(B_100) = 2.368202e-40
P(hit) in box:  A_100 = 1.868182e-70,  B_100 = 1.868182e-70


## Algorithm

The same hit-or-miss procedure from Part 1(a) generalizes directly to $n$ dimensions:

1. Pick a bounding box $\Omega \supset S$ with known volume $|\Omega|$.
2. Draw $N$ points uniformly at random from $\Omega$.
3. Count the hits $M = \#\{i : X_i \in S\}$.
4. Estimate $\hat{V} = |\Omega| \cdot \dfrac{M}{N}$.

For $A_n$ the bounding box is $[0,1]^n$ (volume 1); for $B_n$ it is $[-1,1]^n$ (volume $2^n$). The membership test is the same Euclidean-norm check as in Part 1(a), just over 100 coordinates instead of 2.

## Estimating $V(A_{100})$ and $V(B_{100})$
We apply the same hit-or-miss algorithm from Part 1(a) with $N = 10^7$ samples in batches of $10^5$.


In [23]:
from scipy.stats import norm

CENTER_A = 0.5
RADIUS_SQ_A = 0.25

CHUNK_100 = 100_000
N_TOTAL_100 = 10_000_000


def mc_volume_A(n_dim, N, rng):
    """Uniform samples in [0,1]^n; A_n = {x : ||x - 0.5|| <= 0.5}. Box volume = 1."""
    X = rng.random((N, n_dim))
    inside = np.sum((X - CENTER_A) ** 2, axis=1) <= RADIUS_SQ_A
    M = int(np.sum(inside))
    return M / N, M


def mc_volume_B(n_dim, N, rng):
    """Uniform samples in [-1,1]^n; B_n = {x : ||x|| <= 1}. Box volume = 2^n."""
    X = rng.uniform(-1.0, 1.0, size=(N, n_dim))
    inside = np.sum(X ** 2, axis=1) <= 1.0
    M = int(np.sum(inside))
    box = 2.0 ** n_dim
    return box * (M / N), M


def run_hit_or_miss_at_n(n_dim, N_total, chunk, rng):
    """Batched hit-or-miss for A_n and B_n; returns dict with hits and estimates."""
    out = {}
    for key, fn, box in [("A", mc_volume_A, 1.0), ("B", mc_volume_B, 2.0 ** n_dim)]:
        M_tot, N_done = 0, 0
        while N_done < N_total:
            k = min(chunk, N_total - N_done)
            _, M = fn(n_dim, k, rng)
            M_tot += M
            N_done += k
        p_hat = M_tot / N_done
        est = box * p_hat
        out[key] = {"M": M_tot, "N": N_done, "p_hat": p_hat, "est": est, "box": box}
    return out


def wald_ci_volume(M, N, box_volume, alpha=0.05):
    """Wald CI for volume estimate; returns (lo, hi, se, p_hat)."""
    p_hat = M / N
    if M == 0 or M == N:
        return 0.0, 0.0, 0.0, p_hat
    z = norm.ppf(1 - alpha / 2)
    se_p = np.sqrt(p_hat * (1 - p_hat) / N)
    lo = max(0.0, (p_hat - z * se_p) * box_volume)
    hi = (p_hat + z * se_p) * box_volume
    return lo, hi, se_p * box_volume, p_hat


In [ ]:
print(f"\n=== Hit-or-miss at n={N_DIM} (N={N_TOTAL_100:,}, chunk={CHUNK_100:,}, seed={RNG_SEED}) ===\n")

res_100 = run_hit_or_miss_at_n(N_DIM, N_TOTAL_100, CHUNK_100, RNG)

for label, true_vol, p_true in [
    ("A", TRUE_VOLUME_A100, P_HIT_A100),
    ("B", TRUE_VOLUME_B100, P_HIT_B100),
]:
    r = res_100[label]
    M, N, est = r["M"], r["N"], r["est"]
    exp_M = N * p_true
    print(f"{label}_100:")
    print(f"  True volume       = {true_vol:.6e}")
    print(f"  Hit probability   = {p_true:.6e}")
    print(f"  Expected hits E[M]= {exp_M:.6e}")
    print(f"  Observed M        = {M:,} / {N:,}")
    print(f"  Estimate V_hat    = {est:.6e}")
    if M > 0:
        lo, hi, se, _ = wald_ci_volume(M, N, r["box"])
        rel = abs(est - true_vol) / true_vol * 100
        print(f"  Relative error    = {rel:.4f}%")
        print(f"  95% Wald CI       = [{lo:.6e}, {hi:.6e}]  (SE={se:.6e})")
    else:
        print("  M = 0 -> V_hat = 0; Wald CI degenerates to [0, 0]")
    print()



=== Hit-or-miss at n=100 (N=10,000,000, chunk=100,000, seed=42) ===



## Challenges

The algorithm itself is unchanged from Part 1(a), but the geometry of high dimensions makes it impractical.

**Hit rate significantly decreases.** In 100 dimensions the fraction of the bounding box occupied by the ball is astronomically small. For $A_{100}$ the hit probability is $p_A = V(A_{100}) \approx 10^{-70}$. For $B_{100}$ it is $p_B = V(B_{100})/2^{100} \approx 10^{-60}$. With $N = 10^7$ we expect roughly $1.9\times10^{-63}$ hits — so we virtually always observe $M = 0$. When $M = 0$ the estimator outputs $\hat{V} = 0$, which is useless.

**Degenerate confidence intervals.** The Wald interval collapses to a point when $M = 0$: $\hat{p} = 0$ gives $\mathrm{SE} = 0$, so the reported 95% CI is $[0, 0]$. This hides the fact that we have essentially no information about the true volume.

In [ ]:
from scipy.stats import qmc
# Variance reduction implementations
def mc_volume_A_stratified(n, N, random_state):

    """Latin Hypercube Sampling for A_n."""
    sampler = qmc.LatinHypercube(d=n, seed=random_state)
    X = sampler.random(n=N)
    c = 0.5
    r2 = 0.25
    dist2 = np.sum((X - c) ** 2, axis=1)
    inside = dist2 <= r2
    M = int(np.sum(inside))
    return M / N, M
def mc_volume_B_stratified(n, N, random_state):

    """Latin Hypercube Sampling for B_n."""
    sampler = qmc.LatinHypercube(d=n, seed=random_state)
    X_unit = sampler.random(n=N)
    X = 2.0 * X_unit - 1.0
    dist2 = np.sum(X * X, axis=1)
    inside = dist2 <= 1.0
    M = int(np.sum(inside))
    box_vol = 2.0**n
    return box_vol * (M / N), M
def mc_volume_A_antithetic(n, N, random_state):

    """Antithetic variates for A_n: pair X with 1-X."""
    N_half = N // 2
    X = random_state.random((N_half, n))
    X_anti = 1.0 - X
    X_combined = np.vstack([X, X_anti])
    c = 0.5
    r2 = 0.25
    dist2 = np.sum((X_combined - c) ** 2, axis=1)
    inside = dist2 <= r2
    M = int(np.sum(inside))
    return M / len(X_combined), M
def mc_volume_B_antithetic(n, N, random_state):

    """Antithetic variates for B_n: pair X with -X."""
    N_half = N // 2
    X = random_state.uniform(-1.0, 1.0, size=(N_half, n))
    X_anti = -X
    X_combined = np.vstack([X, X_anti])
    dist2 = np.sum(X_combined * X_combined, axis=1)
    inside = dist2 <= 1.0
    M = int(np.sum(inside))
    box_vol = 2.0**n
    return box_vol * (M / len(X_combined)), M
print("Variance reduction methods loaded.")


In [ ]:
# === Data: MC + VR failure as dimension increases (30 trials, both A_n and B_n) ===
import os
os.makedirs('1', exist_ok=True)

dims_to_test = list(range(2, 21))
N_vr = 10_000_000
CHUNK = 200_000
N_TRIALS = 30

def _rel_err(M, N, true_val):
    if M == 0:
        return 100.0
    return abs(M / N - true_val) / true_val * 100

rows_A, rows_B = [], []
stop_A, stop_B = False, False

for n in dims_to_test:
    true_A = volume_n_ball(n, 0.5)
    true_B = volume_n_ball(n, 1.0)
    box_B = 2.0 ** n
    rels_A = {'Naive': [], 'Stratified': [], 'Antithetic': []}
    rels_B = {'Naive': [], 'Stratified': [], 'Antithetic': []}

    for trial in range(N_TRIALS):
        trial_rng = np.random.default_rng(RNG_SEED + trial)

        # --- A_n ---
        if not stop_A:
            M_naive_A, M_strat_A, M_anti_A = 0, 0, 0
            N_done = 0
            while N_done < N_vr:
                k = min(CHUNK, N_vr - N_done)
                _, M_b = mc_volume_A(n, k, trial_rng)
                M_naive_A += M_b
                N_done += k
            N_done = 0
            while N_done < N_vr:
                k = min(CHUNK, N_vr - N_done)
                _, M_b = mc_volume_A_stratified(n, k, trial_rng)
                M_strat_A += M_b
                N_done += k
            N_done = 0
            while N_done < N_vr:
                k = min(CHUNK, N_vr - N_done)
                _, M_b = mc_volume_A_antithetic(n, k, trial_rng)
                M_anti_A += M_b
                N_done += k
            rels_A['Naive'].append(_rel_err(M_naive_A, N_vr, true_A))
            rels_A['Stratified'].append(_rel_err(M_strat_A, N_vr, true_A))
            rels_A['Antithetic'].append(_rel_err(M_anti_A, N_vr, true_A))
            if M_naive_A == 0 and M_strat_A == 0 and M_anti_A == 0:
                for _ in range(trial + 1, N_TRIALS):
                    for k in rels_A: rels_A[k].append(100.0)
                stop_A = True

        # --- B_n ---
        if not stop_B:
            M_naive_B, M_strat_B, M_anti_B = 0, 0, 0
            N_done = 0
            while N_done < N_vr:
                k = min(CHUNK, N_vr - N_done)
                _, M_b = mc_volume_B(n, k, trial_rng)
                M_naive_B += M_b
                N_done += k
            N_done = 0
            while N_done < N_vr:
                k = min(CHUNK, N_vr - N_done)
                _, M_b = mc_volume_B_stratified(n, k, trial_rng)
                M_strat_B += M_b
                N_done += k
            N_done = 0
            while N_done < N_vr:
                k = min(CHUNK, N_vr - N_done)
                _, M_b = mc_volume_B_antithetic(n, k, trial_rng)
                M_anti_B += M_b
                N_done += k
            rels_B['Naive'].append(_rel_err(M_naive_B, N_vr, true_B / box_B))
            rels_B['Stratified'].append(_rel_err(M_strat_B, N_vr, true_B / box_B))
            rels_B['Antithetic'].append(_rel_err(M_anti_B, N_vr, true_B / box_B))
            if M_naive_B == 0 and M_strat_B == 0 and M_anti_B == 0:
                for _ in range(trial + 1, N_TRIALS):
                    for k in rels_B: rels_B[k].append(100.0)
                stop_B = True

    row_A = {'n': n}
    row_B = {'n': n}
    for method in ['Naive', 'Stratified', 'Antithetic']:
        row_A[method] = np.mean(rels_A[method])
        row_B[method] = np.mean(rels_B[method])
    rows_A.append(row_A)
    rows_B.append(row_B)

    mA = ' / '.join(f"{m}={row_A[m]:6.2f}%" for m in ['Naive','Stratified','Antithetic'])
    mB = ' / '.join(f"{m}={row_B[m]:6.2f}%" for m in ['Naive','Stratified','Antithetic'])
    print(f"n={n:2d}  A: {mA}")
    print(f"       B: {mB}")

    if stop_A and stop_B:
        break

df_A = pd.DataFrame(rows_A)
df_B = pd.DataFrame(rows_B)
print(f"\nData collected: A_n ({len(rows_A)} dims), B_n ({len(rows_B)} dims)")


In [ ]:
# Plot: A_n relative error
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(df_A['n'], df_A['Naive'], 'o-', label='Naive MC', color='#1f77b4', markersize=8, linewidth=2)
ax.plot(df_A['n'], df_A['Stratified'], 's-', label='Stratified (LHS)', color='#2ca02c', markersize=8, linewidth=2)
ax.plot(df_A['n'], df_A['Antithetic'], '^-', label='Antithetic', color='#d62728', markersize=8, linewidth=2)
ax.set_xlabel('Dimension $n$', fontsize=20)
ax.set_ylabel('Relative Error (%)', fontsize=20)
ax.set_title(f'MC and VR Methods vs Dimension ($N = 10^7$, $A_n$, {N_TRIALS} trials)', fontsize=18)
ax.tick_params(axis='both', which='major', labelsize=20)
ax.legend(fontsize=16)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig('1/mc_vr_failure_A.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved to 1/mc_vr_failure_A.png")

In [ ]:
from scipy.stats import gaussian_kde
N_kde = 1_000_00
N_TRIALS_KDE = 5
FIXED_N = 6

true_val = volume_n_ball(FIXED_N, 0.5)
results = {'Naive': [], 'Stratified': [], 'Antithetic': []}

for t in range(N_TRIALS_KDE):
    rng_t = np.random.default_rng(t)

    X = rng_t.random((N_kde, FIXED_N))
    M = np.sum(np.sum((X - 0.5)**2, axis=1) <= 0.25)
    results['Naive'].append(M / N_kde)

    sampler = qmc.LatinHypercube(d=FIXED_N, seed=rng_t)
    X = sampler.random(n=N_kde)
    M = np.sum(np.sum((X - 0.5)**2, axis=1) <= 0.25)
    results['Stratified'].append(M / N_kde)

    X_half = rng_t.random((N_kde // 2, FIXED_N))
    X = np.vstack([X_half, 1.0 - X_half])
    M = np.sum(np.sum((X - 0.5)**2, axis=1) <= 0.25)
    results['Antithetic'].append(M / len(X))

fig, ax = plt.subplots(figsize=(10, 6))
colors = {'Naive': '#1f77b4', 'Stratified': '#2ca02c', 'Antithetic': '#d62728'}
x_grid = np.linspace(true_val - 0.012, true_val + 0.012, 500)

for method in ['Naive', 'Stratified', 'Antithetic']:
    kde = gaussian_kde(results[method])
    ax.plot(x_grid, kde(x_grid), linewidth=2.5, color=colors[method], label=method if method == 'Naive' else f'{method} (std={np.std(results[method]):.5f})')
    ax.fill_between(x_grid, kde(x_grid), alpha=0.15, color=colors[method])

ax.axvline(true_val, color='black', linestyle='--', linewidth=2, label=f'True = {true_val:.6f}')
ax.set_xlabel('Estimate $\\hat{V}(A_6)$', fontsize=20)
ax.set_ylabel('Density', fontsize=20)
ax.set_title(f'Distribution of Estimates ($n={FIXED_N}$, $N=10^6$, {N_TRIALS_KDE} trials)', fontsize=20)
ax.tick_params(axis='both', which='major', labelsize=18)
ax.legend(fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('1/vr_bell.png', dpi=150, bbox_inches='tight')
plt.close()

for m in results:
    arr = np.array(results[m])
    print(f"{m:12s}: mean={arr.mean():.6f}, std={arr.std():.6f}")

In [ ]:
from scipy.stats import gaussian_kde
N_kde = 1_000_00
N_TRIALS_KDE = 5
FIXED_N = 8

true_val = volume_n_ball(FIXED_N, 0.5)
results = {'Naive': [], 'Stratified': [], 'Antithetic': []}

for t in range(N_TRIALS_KDE):
    rng_t = np.random.default_rng(t)

    X = rng_t.random((N_kde, FIXED_N))
    M = np.sum(np.sum((X - 0.5)**2, axis=1) <= 0.25)
    results['Naive'].append(M / N_kde)

    sampler = qmc.LatinHypercube(d=FIXED_N, seed=rng_t)
    X = sampler.random(n=N_kde)
    M = np.sum(np.sum((X - 0.5)**2, axis=1) <= 0.25)
    results['Stratified'].append(M / N_kde)

    X_half = rng_t.random((N_kde // 2, FIXED_N))
    X = np.vstack([X_half, 1.0 - X_half])
    M = np.sum(np.sum((X - 0.5)**2, axis=1) <= 0.25)
    results['Antithetic'].append(M / len(X))

fig, ax = plt.subplots(figsize=(10, 6))
colors = {'Naive': '#1f77b4', 'Stratified': '#2ca02c', 'Antithetic': '#d62728'}
x_grid = np.linspace(true_val - 0.012, true_val + 0.012, 500)

for method in ['Naive', 'Stratified', 'Antithetic']:
    kde = gaussian_kde(results[method])
    ax.plot(x_grid, kde(x_grid), linewidth=2.5, color=colors[method], label=method if method == 'Naive' else f'{method} (std={np.std(results[method]):.5f})')
    ax.fill_between(x_grid, kde(x_grid), alpha=0.15, color=colors[method])

ax.axvline(true_val, color='black', linestyle='--', linewidth=2, label=f'True = {true_val:.6f}')
ax.set_xlabel('Estimate $\\hat{V}(A_6)$', fontsize=20)
ax.set_ylabel('Density', fontsize=20)
ax.set_title(f'Distribution of Estimates ($n={FIXED_N}$, $N=10^6$, {N_TRIALS_KDE} trials)', fontsize=20)
ax.tick_params(axis='both', which='major', labelsize=18)
ax.legend(fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('1/vr_bell.png', dpi=150, bbox_inches='tight')
plt.close()

for m in results:
    arr = np.array(results[m])
    print(f"{m:12s}: mean={arr.mean():.6f}, std={arr.std():.6f}")

In [ ]:
from scipy.stats import gaussian_kde
N_kde = 1_000_000
N_TRIALS_KDE = 5
FIXED_N = 10

true_val = volume_n_ball(FIXED_N, 0.5)
results = {'Naive': [], 'Stratified': [], 'Antithetic': []}

for t in range(N_TRIALS_KDE):
    rng_t = np.random.default_rng(t)

    X = rng_t.random((N_kde, FIXED_N))
    M = np.sum(np.sum((X - 0.5)**2, axis=1) <= 0.25)
    results['Naive'].append(M / N_kde)

    sampler = qmc.LatinHypercube(d=FIXED_N, seed=rng_t)
    X = sampler.random(n=N_kde)
    M = np.sum(np.sum((X - 0.5)**2, axis=1) <= 0.25)
    results['Stratified'].append(M / N_kde)

    X_half = rng_t.random((N_kde // 2, FIXED_N))
    X = np.vstack([X_half, 1.0 - X_half])
    M = np.sum(np.sum((X - 0.5)**2, axis=1) <= 0.25)
    results['Antithetic'].append(M / len(X))

fig, ax = plt.subplots(figsize=(10, 6))
colors = {'Naive': '#1f77b4', 'Stratified': '#2ca02c', 'Antithetic': '#d62728'}
x_grid = np.linspace(true_val - 0.012, true_val + 0.012, 500)

for method in ['Naive', 'Stratified', 'Antithetic']:
    kde = gaussian_kde(results[method])
    ax.plot(x_grid, kde(x_grid), linewidth=2.5, color=colors[method], label=method if method == 'Naive' else f'{method} (std={np.std(results[method]):.5f})')
    ax.fill_between(x_grid, kde(x_grid), alpha=0.15, color=colors[method])

ax.axvline(true_val, color='black', linestyle='--', linewidth=2, label=f'True = {true_val:.6f}')
ax.set_xlabel('Estimate $\\hat{V}(A_6)$', fontsize=20)
ax.set_ylabel('Density', fontsize=20)
ax.set_title(f'Distribution of Estimates ($n={FIXED_N}$, $N=10^6$, {N_TRIALS_KDE} trials)', fontsize=20)
ax.tick_params(axis='both', which='major', labelsize=18)
ax.legend(fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('1/vr_bell.png', dpi=150, bbox_inches='tight')
plt.close()

for m in results:
    arr = np.array(results[m])
    print(f"{m:12s}: mean={arr.mean():.6f}, std={arr.std():.6f}")

## Analysis of Variance Reduction Results

**Key Findings:**

1. **Stratified Sampling (Latin Hypercube)**: improvement at low dimensions
   - Provides variance reduction across all dimensions
   - Works by ensuring uniform coverage of the sampling space (McKay et al., 1979)
   - Most reliable method with minimal tuning required

2. **Antithetic Variates**: Fail even at low dimensions
   - Effective for symmetric regions in theory (Hammersley & Morton, 1956), but showed variance *increase* in our experiments
   - The ball indicator function is not monotone in the sampling coordinates, violating the key condition for antithetic variance reduction


**Why these methods still fail at n=100:**

Both methods reduce variance by constant factors, but the curse of dimensionality is exponential. At n=100 the hit probability is $p \approx 10^{-70}$, requiring $N \approx 10^{70}$ samples. A constant-factor speedup is fundamentally insufficient.


## Geometric Explanation: Why Variance Reduction Fails at n=100

The previous section showed that stratified and antithetic methods fail at $n=100$ even though stratified works at moderate dimensions. The root cause is geometric: the **thin shell phenomenon**.

**Theorem (Concentration of the norm, Vershynin 2025, Theorem 3.1.1, p.59).** Let $g \sim N(0, \sigma^2 I_n)$. Then $\|g\|_2$ concentrates tightly around $\sigma\sqrt{n}$:

$$P\bigl(\bigl|\|g\|_2 - \sigma\sqrt{n}\bigr| \geq t\bigr) \leq 2\exp\!\left(-\frac{ct^2}{\sigma^4}\right).$$

This theorem reveals why uniform sampling fails: in high dimensions, almost all of the ball's volume concentrates in a thin shell near the surface. Over 99% of the volume of the unit ball in $\mathbb{R}^n$ lies within distance $5/n$ from the surface (Vershynin, 2025, Exercise 0.7, p.7). At $n=100$, this shell has thickness just $0.05$ around radius $0.5$ for $A_{100}$ (or $0.05$ around radius $1$ for $B_{100}$).


### Observations

- **Stratified sampling (LHS)**: Provides modest improvement at low dimensions but cannot prevent failure as dimension grows.
- **Antithetic variates**: Does not help -- performs similarly to naive MC across all dimensions.
- **All three methods** converge to 100% relative error at similar dimensions, confirming that constant-factor variance reduction cannot overcome the exponential geometry.

The histogram confirms Theorem 3.1.1 (p.59): $\|g - c\|_2$ concentrates in a narrow band around $\sigma\sqrt{n}$ with constant-order variance (independent of $n$). At $n=100$:

- For $A_{100}$ ($\sigma=0.055$): the peak is at $0.55$, just above the ball radius $R=0.5$, so roughly half the proposal samples land inside the ball.
- For $B_{100}$ ($\sigma=0.11$): the peak is at $1.10$, just above $R=1.0$.

**Key insight**: The Gaussian proposal naturally places its mass on the thin shell where the ball lives, while uniform sampling from the bounding box misses it entirely. The importance weights correct for the non-uniform proposal, making the estimator efficient at $n=100$.

## Importance Sampling at $n=100$

We now implement importance sampling with a Gaussian proposal distribution.

**Method**: Sample from $q(x) = N(c, \sigma^2 I_n)$ instead of uniform. Use theoretical $\sigma = R/\sqrt{n}$:
- For $A_{100}$: $\sigma = 0.5/\sqrt{100} = 0.05$
- For $B_{100}$: $\sigma = 1.0/\sqrt{100} = 0.10$

**Standard IS estimator**:

$$\hat{V} = \frac{1}{N} \sum_{i=1}^N \frac{\mathbb{1}_{\{x_i \in S\}}}{q(x_i)} \cdot |\Omega|$$

where:
- $x_i \sim q(x) = N(c, \sigma^2 I_n)$ (Gaussian samples)
- $q(x_i) = \frac{1}{(2\pi\sigma^2)^{n/2}} \exp\left(-\frac{\|x_i-c\|^2}{2\sigma^2}\right)$ (Gaussian density)
- Weight: $w_i = \frac{1}{q(x_i)}$ for points inside the ball
- Estimate: $\hat{V} = \text{mean}(w_i) \times |\Omega|$

In [ ]:
N_IS = 10_000_000
N_IS_TRIALS = 1
CHUNK_IS = 100_000


def is_estimate_A_batched(n_dim, N, sigma, rng, chunk=CHUNK_IS):
    """Batched IS for A_n: returns weighted mean."""
    center = np.full(n_dim, 0.5)
    R2 = 0.25
    sum_w = 0.0
    N_done = 0
    while N_done < N:
        k = min(chunk, N - N_done)
        X = rng.normal(loc=center, scale=sigma, size=(k, n_dim))
        d2 = np.sum((X - center) ** 2, axis=1)
        inside = d2 <= R2
        in_box = np.all((X >= 0) & (X <= 1), axis=1)
        valid = inside & in_box
        log_q = -0.5 * n_dim * np.log(2 * np.pi * sigma**2) - d2 / (2 * sigma**2)
        w = np.where(valid, np.exp(-log_q), 0.0)
        sum_w += np.sum(w)
        N_done += k
    return sum_w / N


def is_estimate_B_batched(n_dim, N, sigma, rng, chunk=CHUNK_IS):
    """Batched IS for B_n: returns weighted mean."""
    R2 = 1.0
    sum_w = 0.0
    N_done = 0
    while N_done < N:
        k = min(chunk, N - N_done)
        X = rng.normal(loc=0.0, scale=sigma, size=(k, n_dim))
        d2 = np.sum(X ** 2, axis=1)
        inside = d2 <= R2
        in_box = np.all((X >= -1) & (X <= 1), axis=1)
        valid = inside & in_box
        log_q = -0.5 * n_dim * np.log(2 * np.pi * sigma**2) - d2 / (2 * sigma**2)
        w = np.where(valid, np.exp(-log_q), 0.0)
        sum_w += np.sum(w)
        N_done += k
    return sum_w / N


sigma_A = 0.5 / np.sqrt(N_DIM)
sigma_B = 1.0 / np.sqrt(N_DIM)

print(f"=== Importance sampling at n={N_DIM} (seed={RNG_SEED}) ===")
print(f"sigma_A = {sigma_A:.4f}, sigma_B = {sigma_B:.4f}")
print(f"N = {N_IS:,} per trial, {N_IS_TRIALS} trial(s), chunk={CHUNK_IS:,}\n")

ests_A, ests_B = [], []
for t in range(N_IS_TRIALS):
    est_A = is_estimate_A_batched(N_DIM, N_IS, sigma_A, RNG)
    est_B = is_estimate_B_batched(N_DIM, N_IS, sigma_B, RNG)
    ests_A.append(est_A)
    ests_B.append(est_B)

mean_A, mean_B = np.mean(ests_A), np.mean(ests_B)
rel_A = abs(mean_A - TRUE_VOLUME_A100) / TRUE_VOLUME_A100 * 100
rel_B = abs(mean_B - TRUE_VOLUME_B100) / TRUE_VOLUME_B100 * 100

if N_IS_TRIALS > 1:
    std_A = np.std(ests_A, ddof=1)
    std_B = np.std(ests_B, ddof=1)
    print(f"A_100: true={TRUE_VOLUME_A100:.6e}, est={mean_A:.6e}, rel_err={rel_A:.2f}%, std={std_A:.6e}")
    print(f"B_100: true={TRUE_VOLUME_B100:.6e}, est={mean_B:.6e}, rel_err={rel_B:.2f}%, std={std_B:.6e}")
else:
    print(f"A_100: true={TRUE_VOLUME_A100:.6e}, est={mean_A:.6e}, rel_err={rel_A:.2f}%")
    print(f"B_100: true={TRUE_VOLUME_B100:.6e}, est={mean_B:.6e}, rel_err={rel_B:.2f}%")
print()
print(f"{'Method':<22} {'A_100 err':>12} {'B_100 err':>12}")
print(f"{'Naive MC':<22} {'100% (M=0)':>12} {'100% (M=0)':>12}")
print(f"{'IS (Gaussian q)':<22} {rel_A:>11.2f}% {rel_B:>11.2f}%")